# Student 3 — Image Analyst: Crime/Incident Scene Object Detection

**Assignment:** AI for Engineers — Multimodal Crime / Incident Report Analyzer  
**Role:** Student 3 — Image Analyst  
**Pipeline:** Load images → OpenCV preprocessing → YOLOv8 detection → pytesseract OCR → Scene classification → Export CSV

### Required Output Columns
`Image_ID | Scene_Type | Objects_Detected | Bounding_Boxes | Confidence`

---

## Dataset
**Roboflow Fire Detection** — pre-labeled fire and smoke images, YOLOv8 format  
- Open `universe.roboflow.com`, search fire detection, pick dataset with 1000+ images  
- Download in **YOLOv8 format** (free Roboflow account required)  
- Roboflow also provides a starter Colab notebook — use it to run inference immediately

In [ ]:
# ============================================================
# CELL 1 — Install all dependencies
# ============================================================
!pip install ultralytics opencv-python pytesseract roboflow -q
print('All packages installed.')

In [ ]:
# ============================================================
# CELL 2 — Download Roboflow Fire Detection Dataset
# Replace RF_API_KEY with your free Roboflow API key
# Get it at: roboflow.com → Account Settings → API Key
# ============================================================
from roboflow import Roboflow
import os

RF_API_KEY = "JrpwBu99OhGyhR1Gcqy0"   # <-- replace this

rf = Roboflow(api_key=RF_API_KEY)

# Dataset: Fire Detection (1000+ images, YOLOv8 trained model badge)
project = rf.workspace("mohamedmoustafa").project("fire-detection-or9c6")
version = project.version(2)
dataset = version.download("yolov8")

# Images will be downloaded to: ./fire-detection-or9c6-2/
IMAGE_DIR = os.path.join(dataset.location, 'test', 'images')
print(f'Dataset downloaded. Test images at: {IMAGE_DIR}')
print(f'Sample images: {os.listdir(IMAGE_DIR)[:5]}')

In [ ]:
# ============================================================
# CELL 2b — Alternative: Use local images (no API key needed)
# Place your crime/incident scene images in this folder,
# or use the bundled test images already present.
# ============================================================
import os

# Use images in the current directory
IMAGE_DIR = '.'
image_files = [f for f in os.listdir(IMAGE_DIR)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f'Found {len(image_files)} images: {image_files}')

In [ ]:
# ============================================================
# CELL 3 — OpenCV image preprocessing
# Resize, denoise, enhance contrast for better detection
# ============================================================
import cv2
import numpy as np
import os

PROCESSED_DIR = 'processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

def preprocess_image(img_path, target_size=(640, 640)):
    """Resize, denoise, normalize for YOLOv8 inference."""
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f'Cannot read: {img_path}')
    # Resize to YOLOv8 standard input
    img = cv2.resize(img, target_size)
    # Denoise
    img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    # CLAHE contrast enhancement on L channel
    lab   = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab   = cv2.merge([clahe.apply(l), a, b])
    img   = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return img

processed_paths = {}
for img_file in image_files:
    src = os.path.join(IMAGE_DIR, img_file)
    dst = os.path.join(PROCESSED_DIR, img_file)
    img = preprocess_image(src)
    cv2.imwrite(dst, img)
    processed_paths[img_file] = dst
    print(f'Preprocessed: {img_file} -> {dst}')

In [ ]:
# ============================================================
# CELL 4 — YOLOv8 Object Detection
# yolov8n.pt — COCO pre-trained (detects 80 classes)
# For fire/smoke: use Roboflow's fine-tuned model from Cell 2
# ============================================================
from ultralytics import YOLO

# Standard model (COCO 80-class)
model = YOLO('yolov8n.pt')

# Fire-detection fine-tuned model (downloaded from Roboflow in Cell 2)
model = YOLO(f'{dataset.location}/weights/best.pt')


SCENE_MAP = [
    (['fire', 'smoke'],                              'Fire Scene'),
    (['gun', 'knife', 'weapon', 'pistol'],           'Crime / Assault Scene'),
    (['car', 'truck', 'bus', 'motorcycle',
      'bicycle', 'stop sign', 'traffic light'],      'Road Incident Scene'),
    (['person'],                                     'Public Disturbance Scene'),
]

def classify_scene(objects):
    obj_lower = [o.lower() for o in objects]
    for keywords, label in SCENE_MAP:
        if any(k in obj_lower for k in keywords):
            return label
    return 'General Incident Scene'

detection_results = {}
for img_file, proc_path in processed_paths.items():
    results  = model(proc_path, verbose=False)
    r        = results[0]
    detected, confs, bboxes = [], [], []
    for box in r.boxes:
        cls_name = model.names[int(box.cls[0])]
        conf     = round(float(box.conf[0]), 2)
        xyxy     = box.xyxy[0].tolist()
        detected.append(cls_name)
        confs.append(conf)
        bboxes.append(f'[{int(xyxy[0])},{int(xyxy[1])},{int(xyxy[2])},{int(xyxy[3])}]')
    detection_results[img_file] = {
        'detected': detected, 'confs': confs, 'bboxes': bboxes
    }
    print(f'{img_file}: {dict.fromkeys(detected)} | avg_conf={round(sum(confs)/max(len(confs),1),2)}')

In [ ]:
# ============================================================
# CELL 5 — pytesseract OCR: extract text from images
# Targets: license plates, street signs, banners
# ============================================================
import pytesseract
import cv2

def extract_text_from_image(img_path):
    """Run OCR on image; return any readable text."""
    img  = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Threshold to improve OCR on text regions
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
    text = pytesseract.image_to_string(thresh, config='--psm 11').strip()
    # Clean noise: keep only printable ASCII, min 2 chars
    import re
    text = re.sub(r'[^\x20-\x7E\n]', '', text)
    lines = [l.strip() for l in text.splitlines() if len(l.strip()) >= 2]
    return ' | '.join(lines) if lines else 'none'

ocr_results = {}
for img_file, proc_path in processed_paths.items():
    try:
        ocr_text = extract_text_from_image(proc_path)
    except Exception as e:
        ocr_text = f'OCR error: {e}'
    ocr_results[img_file] = ocr_text
    print(f'{img_file}: OCR -> {ocr_text[:80]}')

In [ ]:
# ============================================================
# CELL 6 — Build structured DataFrame
# ============================================================
import pandas as pd

rows = []
for i, img_file in enumerate(image_files):
    det  = detection_results.get(img_file, {})
    detected = det.get('detected', [])
    confs    = det.get('confs', [])
    bboxes   = det.get('bboxes', [])
    
    img_id     = os.path.splitext(img_file)[0].upper()
    scene_type = classify_scene(detected)
    obj_str    = ', '.join(dict.fromkeys(detected)) if detected else 'none'
    bbox_str   = '; '.join(bboxes) if bboxes else 'none'
    avg_conf   = round(sum(confs) / max(len(confs), 1), 2)
    
    rows.append({
        'Image_ID':         img_id,
        'Scene_Type':       scene_type,
        'Objects_Detected': obj_str,
        'Bounding_Boxes':   bbox_str,
        'Confidence':       avg_conf,
    })

image_df = pd.DataFrame(rows)
print(image_df.to_string(index=False))

In [ ]:
# ============================================================
# CELL 7 — Validate and save image_output.csv
# ============================================================

REQUIRED_COLUMNS = ['Image_ID', 'Scene_Type', 'Objects_Detected', 'Bounding_Boxes', 'Confidence']

image_df = image_df[REQUIRED_COLUMNS]

assert list(image_df.columns) == REQUIRED_COLUMNS, f'Column mismatch: {list(image_df.columns)}'
assert len(image_df) > 0, 'DataFrame is empty!'

image_df.to_csv('image_output.csv', index=False)

print(f'Saved image_output.csv')
print(f'  Rows: {len(image_df)} | Columns: {REQUIRED_COLUMNS}')
print()
print(image_df.to_string(index=False))